This notebook allows you to run ACOLITE processing on Google Colab. The required dependencies are installed and the latest version is cloned from GitHub (https://github.com/acolite/acolite).

In this example, a PlanetScope image is retrieved from ESA Sample Data (https://earth.esa.int/eogateway/missions/planetscope/sample-data), processed and stored on in the Colab runtime environment. Ancillary data can be retrieved if EarthData credentials are set up, but are not essential for this example.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be

The next cell sets up the ACOLITE dependencies in Colab if they are not already there. This is specifical to Colab and you will need to set up an ACOLITE environment on your machine if you want to run it locally.

In [ ]:
# general imports
import os, sys

# install dependencies for acolite
!pip install numpy matplotlib scipy gdal pyproj scikit-image pyhdf pyresample netcdf4 h5py requests pygrib cartopy

The ACOLITE code is cloned from the GitHub repository.

In [ ]:
# clone acolite repo if it does not exist
if not os.path.exists('acolite'):
  !git clone --depth 1 https://github.com/acolite/acolite

The ACOLITE code is imported as "ac" so the Python functions can be called directoy.

In [ ]:
# add acolite to path and import
sys.path.append('acolite')
import acolite as ac

The next cell downloads a sample SuperDove scene kindly provided by ESA.

In [ ]:
## download sample SuperDove scene from ESA
scene_url = 'https://earth.esa.int/eogateway/ftp/missions/sample-data/third-party-missions/planetscope/PSScene_Basic_Analytic_8b_udm2.zip'

# make input directory
if not os.path.exists('Input/'): os.makedirs('Input/')
local_scene_zip = '{}/{}'.format('Input', os.path.basename(scene_url))
local_scene = '{}/{}'.format('Input', os.path.basename(scene_url)[0:-4])

## download scene
if not os.path.exists(local_scene_zip):
  !wget -P 'Input' $scene_url

## unzip file
if not os.path.exists(local_scene):
  !unzip -d 'Input' $local_scene_zip

## find file
import glob
inputfile = glob.glob('{}/**/basic_analytic_8b_udm2'.format(local_scene))[0]


In [ ]:
## location of interest
## here for centering 3x3 km box and extracting spectra
region_name = 'EtangdEntressen'
station_lon = 4.91432
station_lat = 43.60362
station_box_size = 3

The next cell sets up a Python dictionary with typical settings for processing ACOLITE/DSF for a region of interest (here a square box centred on the location from the cell above). DSF processing is based on Vanhellemont (2019) https://doi.org/10.1016/j.rse.2019.03.010

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': inputfile,
            'output': 'Output/DSF/',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': station_box_size,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use DSF processing
            'atmospheric_correction_method': 'dark_spectrum',

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

The next cell passes the settings dictionary to ACOLITE and runs the processing. Note that the first run will be slightly slower, as the LUTs for the current sensor will have to be downloaded. A reprojection of the original data is automatically performed using Copernicus 30m DEM.

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

Now we will change the processing run to use Remote Sensing Adjacency Correction(RAdCor) by Castagna and Vanhellemont (2025) https://doi.org/10.1364/AO.546766

In [ ]:
# acolite settings - RAdCor processing
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': inputfile,
            'output': 'Output/RAdCor/',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 3,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use RAdCor processing
            'atmospheric_correction_method': 'radcor',

            ## add a buffer and crop to centre to avoid masked data around the scene edges
            'limit_buffer': 5,
            'limit_buffer_units': 'km',
            'radcor_crop_centre': True,

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

RAdCor processing will take a bit longer than the DSF as a result of the image (de)convolutions over large spatial extents (5 km PSF radius). The convolutions will be still rather fast due to the use of the frequency domain and Fast Fourier Transforms. Here running the processing after the DSF in the cells above will not re-acquire the LUTs.

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

Below we will plot an RGB composite of the DSF surface reflectance, and compare spectra from DSF, RAdCor, and TOA data.

In [ ]:
## imports for plotting
import glob
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
## create rhos RGB from DSF data
ncf = glob.glob('Output/DSF/*L2R*.nc')[0]

gatts = ac.shared.nc_gatts(ncf)
title = '{} {}'.format(gatts['sensor'].replace('_', '/'), gatts['isodate'][0:19])

lat = ac.shared.nc_data(ncf, 'lat')
lon = ac.shared.nc_data(ncf, 'lon')
datasets = ac.shared.nc_datasets(ncf)

stack = []
waves = []
for ds in datasets:
  if ds.startswith('rhos_'):
      waves.append(float(ds[ds.find('_')+1:]))
      stack.append(ac.shared.nc_data(ncf, ds))
stack = np.asarray(stack)
waves = np.asarray(waves)
rgb = ac.map.rgb.stack(waves, stack,  rgb_max = [0.15] * 3,)

fig, ax = plt.subplots()
plt.xlabel('Longitude (°E)')
plt.ylabel('Latitude (°N)')
plt.title(title)

mesh = plt.pcolormesh(lon, lat, rgb)
plt.plot(station_lon, station_lat, 'x', color = 'Red')
plt.savefig('Output/DSF/RGB.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

The next cell extracts data for a given location (here the ROI centre) for an 11x11 pixel box.

In [ ]:
## extract DSF Rrs spectrum
ncf = glob.glob('Output/DSF/*L2W*.nc')[0]
ext_dsf = ac.shared.nc_extract_point(ncf, station_lon, station_lat, box_size = 11)

## extract DSF rhot and rhos spectra
ncf = glob.glob('Output/DSF/*L2R*.nc')[0]
ext2_dsf = ac.shared.nc_extract_point(ncf, station_lon, station_lat, box_size = 11)

## extract RAdCor Rrs spectrum
ncf = glob.glob('Output/RAdCor/*L2W*.nc')[0]
ext_radcor = ac.shared.nc_extract_point(ncf, station_lon, station_lat, box_size = 11)

## extract RAdCor rhot and rhos spectra
ncf = glob.glob('Output/RAdCor/*L2R*.nc')[0]
ext2_radcor = ac.shared.nc_extract_point(ncf, station_lon, station_lat, box_size = 11)

Now the DSF and RAdCor results are plotted. They align fairly well, but show a significant difference in the NIR, where the contrast between land and water is highest.

In [ ]:
## plot DSF Rrs
title = '{} {}'.format(ext_dsf['gatts']['sensor'].replace('_', '/'), ext_dsf['gatts']['isodate'][0:19])
mean_dsf = np.asarray([np.nanmean(ext_dsf['data'][ds]) for ds in ext_dsf['Rrs_datasets']])
std_dsf = np.asarray([np.nanstd(ext_dsf['data'][ds]) for ds in ext_dsf['Rrs_datasets']])

plt.errorbar(ext_dsf['Rrs_wave'], mean_dsf, marker = '.', linestyle = '--', yerr = std_dsf, label = 'DSF')

## plot RAdCor Rrs
mean_radcor = np.asarray([np.nanmean(ext_radcor['data'][ds]) for ds in ext_radcor['Rrs_datasets']])
std_radcor = np.asarray([np.nanstd(ext_radcor['data'][ds]) for ds in ext_radcor['Rrs_datasets']])

plt.errorbar(ext_radcor['Rrs_wave'], mean_radcor, marker = '.', linestyle = '--', yerr = std_radcor, label = 'RAdCor')


plt.xlabel('Wavelength (nm)')
plt.ylabel(r'Rrs ($sr^{-1}$)')
plt.ylim(-0.002, 0.061)

plt.legend()

plt.title(title)

plt.savefig('Output/Rrs.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

Finally, the TOA spectrum (here divided by pi to match the Remote sensing reflectance) is added, showing the impact of the atmospheric correction.

In [ ]:
## plot Rrs and rhot (/pi)
title = '{} {}'.format(ext_dsf['gatts']['sensor'].replace('_', '/'), ext_dsf['gatts']['isodate'][0:19])
meant = np.asarray([np.nanmean(ext2_dsf['data'][ds]) for ds in ext2_dsf['rhot_datasets']])
stdt = np.asarray([np.nanstd(ext2_dsf['data'][ds]) for ds in ext2_dsf['rhot_datasets']])

plt.errorbar(ext2_dsf['rhot_wave'], meant/np.pi, marker = '.', linestyle = '--', yerr = stdt/np.pi, label = 'TOA', color ='Black')
plt.errorbar(ext_dsf['Rrs_wave'], mean_dsf, marker = '.', linestyle = '--', yerr = std_dsf, label = 'Rrs DSF')
plt.errorbar(ext_radcor['Rrs_wave'], mean_radcor, marker = '.', linestyle = '--', yerr = std_radcor, label = 'Rrs RAdCor')

plt.xlabel('Wavelength (nm)')
plt.ylabel(r'Rrs ($sr^{-1}$)')
plt.ylim(-0.002, 0.061)

plt.legend()
plt.title(title)

plt.savefig('Output/Rrs_TOA.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

Files generated with this Notebook are in the "Files" section to the left, and will be deleted when the Notebook is inactive for too long. You can download the processed files and PNG plots.